================================================================================================================
The following is to calculate first component of SH residuals to non-depression-related SH for **GP1**
================================================================================================================
The training and test sets are the same from GLM
================================================================================================================
Age and sex effects have already been removed in GLM setp
================================================================================================================

In [ ]:
import sys
sys.path.append('working_path')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity

In [ ]:
# Initiate a dictionary
dataframes_train = {}
dataframes_test = {}

# Loop the dataframes and store in the dictionary
for i in range(1, 6):
    path_train = f'save_path/SH_residuals_training_fold_{i}.csv'
    path_test = f'save_path/SH_residuals_test_fold_{i}.csv'

    dataframes_train[f'df_SH_res_train_{i}'] = pd.read_csv(path_train)
    dataframes_test[f'df_SH_res_test_{i}'] = pd.read_csv(path_test)

# Get the global info for all dfs
globals().update(dataframes_train)
globals().update(dataframes_test)

In [ ]:
# Define original variables
sleep = ['sleep_variables']

res_cols = [f"{col}_res" for col in sleep]

# Define scaling
scale = StandardScaler()

In [ ]:
# Loop through all dfs
for fold in range(1, 6):
    df_res_test = globals()[f'df_SH_res_test_{fold}']
    df_SH_res_train = globals()[f'df_SH_res_train_{fold}'][res_cols]
    df_SH_res_test = globals()[f'df_SH_res_test_{fold}'][res_cols]

    # Scaling on training set and project to test set
    df_SH_res_train_s = scale.fit_transform(df_SH_res_train)
    df_SH_res_test_s = scale.transform(df_SH_res_test)
    
    # Run PCA to get eigenvalues
    fa = FactorAnalyzer(rotation='varimax', method='principal')
    fa.fit(df_SH_res_train_s)
    ev_SH, v_SH = fa.get_eigenvalues()

    # Determine number of components with eigenvalue > 1
    n_components_SH = sum(ev_SH > 1)
    print(f"Number of components for SH with eigenvalue > 1 in fold {fold}: {n_components_SH}")

    # Get rotated loadings for SH
    loadings_SH = pd.DataFrame(fa.loadings_, index=res_cols)
    print(f"Rotated component loadings for SH in fold {fold}:\n", loadings_SH)

    # Get variance explained by each component for SH
    variance_SH = fa.get_factor_variance()
    explained_variance_SH_df = pd.DataFrame({
        'SS Loadings': variance_SH[0],
        'Proportion Var': variance_SH[1],
        'Cumulative Var': variance_SH[2]
    }, index=[f'PC{i+1}' for i in range(len(variance_SH[0]))])
    print("\nExplained Variance:\n", explained_variance_SH_df)

    # Transform to ML data
    SH_scores = fa.transform(df_SH_res_test_s)
    
    # Plot the loadings for SH
    plt.figure(figsize=(10, 6))
    sns.heatmap(loadings_SH, annot=True, cmap='viridis', cbar=True, center=0)
    plt.title(f'Rotated Component Loadings for SH in fold {fold}')
    plt.xlabel('Components')
    plt.ylabel('SH')
    plt.tight_layout()
    plt.show()

    # Only save PC1 to the test set
    df_PC1 = df_res_test.copy()
    df_PC1['pca_comp1_SH'] = SH_scores[:, 0]
    df_PC1.to_csv(f'working_path/PCA_PC1_SH_residuals_fold_{fold}.csv', index=False)